# EcoFOCIpy PostQC example to update netcdf file

see [EcoFOCIpy_sbe_UAF.ipynb](EcoFOCIpy_sbe_UAF.ipynb) for initial processing and formatting.  Data would have been convertted from raw instrument format to csv files, initial dirty plots created and a meta-data rich and meta-data poor dataset would have been created.  The metadata poor csv file is used for the initial preliminary data archive, the metadata rich netcdf file is used for QC procedures and eventual final data archive.

## Next Steps

QC of data (plot parameters with other instruments)
- be sure to updated the qc_status and the history

- **TODO** Programmatically simplify following tools

In [122]:
%matplotlib inline
import matplotlib.pyplot as plt
import yaml

import pandas as pd
import xarray as xr
import datetime
import EcoFOCIpy.io.erddap as erddap

import EcoFOCIpy.metaconfig.load_config as load_config

In [123]:
sample_data_dir = '/Users/bell/Programs/EcoFOCIpy/'
user_data_dir = '/Users/bell/ecoraid/2025/Moorings/25bs8a/working/'
filename = '25bs8a_s37_0054m.nc'

### Post Manual QC load data and rebuild nc file

- using excel for a few points or a dynamic web map for multiple (a tool on ecofoci-field.pmel.noaa.gov) or any other method to identify spikes
- remove values above 32psu and below 30psu

**NOTE** if you use excel, be cautious about how your time is formatted (its important) - use custom formatting and make it of the form `yyyy-mm-ddTHH:MM:SSZ` to assure proper read in to xarray

*TODO:* use erddap hosted data 

In [124]:
# this loads the initial working netcdf file
wop_nc_qc = xr.load_dataset(user_data_dir+filename)

In [125]:
wop_nc_qc

<xarray.Dataset> Size: 280kB
Dimensions:       (time: 8752, depth: 1, latitude: 1, longitude: 1)
Coordinates:
  * time          (time) datetime64[ns] 70kB 2025-08-15T02:00:00 ... 2026-08-...
  * depth         (depth) float64 8B 54.0
  * latitude      (latitude) float64 8B 62.2
  * longitude     (longitude) float64 8B -174.7
Data variables:
    temperature   (time, depth, latitude, longitude) float64 70kB -1.446 ... ...
    conductivity  (time, depth, latitude, longitude) float64 70kB 2.553 ... 2...
    salinity      (time, depth, latitude, longitude) float64 70kB 31.83 ... 3...
Attributes: (12/33)
    Latitude_DegMMddW:                      62 12.106 N
    Longitude_DegMMddN:                     174 39.673 W
    MooringID:                              25BS-8A
    platform_deployment_date:               2025-08-15T01:18:00Z
    platform_deployment_cruise_name:        SKQ202515S
    platform_recovery_date:                 2026-08-14T17:00:01Z
    ...                                     ...
    InstrumentSerialNumber:                 1858
    InstrumentType:                         SBE-37
    date_created:                           2026-09-13T16:23:26Z
    date_modified:                          
    QC_indicator:                           unknown
    history:                                Trimmed to deployment.

In [126]:
# this reads the previous csv and assumes you've made modifications to the content but not the structure (record length or variable names)
df_qc = pd.read_csv(user_data_dir+filename.replace('nc','to_edit.csv'), index_col=['time','depth','latitude','longitude']) #order is important

In [127]:
df_qc

,,,,temperature,conductivity,salinity
time,depth,latitude,longitude,,,
2025-08-15T02:00:00Z,54,62.201767,-174.661217,-1.4463,2.55290,31.833216
2025-08-15T03:00:00Z,54,62.201767,-174.661217,-1.4463,2.55256,31.828557
2025-08-15T04:00:00Z,54,62.201767,-174.661217,-1.4474,2.55262,31.830554
2025-08-15T05:00:00Z,54,62.201767,-174.661217,-1.4461,2.55280,31.831632
2025-08-15T06:00:00Z,54,62.201767,-174.661217,-1.4481,2.55182,31.820341
...,...,...,...,...,...,...
2026-08-14T13:00:00Z,54,62.201767,-174.661217,-1.2253,2.56674,31.786766
2026-08-14T14:00:00Z,54,62.201767,-174.661217,-1.2126,2.56680,31.774093
2026-08-14T15:00:00Z,54,62.201767,-174.661217,-1.2133,2.56659,31.771979


In [128]:
#replace original data with editied data
for var in wop_nc_qc.variables:
    if var not in ['time','latitude','longitude','depth'] + []:
        print(var)
        try:
            wop_nc_qc[var].values=xr.Dataset.from_dataframe(df_qc)[var]
            print(f'{var} success')
        except:
            pass

temperature
temperature success
conductivity
conductivity success
salinity
salinity success


In [129]:
# for var in ['latitude','longitude','depth']:
#     print(var)
#     try:
#         wop_nc_qc.coords[var]=xr.Dataset.from_dataframe(df_qc)[var]
#         print(f'{var} success')
#     except:
#         pass

In [130]:
#drop oxy conc for oxygen
# wop_nc_qc = wop_nc_qc.drop('oxy_conc')

### Update global attributes

In [131]:
wop_nc_qc.attrs.update({'QC_indicator': 'ProbablyGood'})
try:
    wop_nc_qc.attrs.update({'history':(wop_nc_qc.history + "\nQC'd: "+ str(datetime.datetime.today()))})
except:
    pass

In [132]:
wop_nc_qc.attrs.update({'date_modified':str(datetime.datetime.today())})

In [133]:
wop_nc_qc.to_netcdf(user_data_dir+filename,format="NETCDF3_CLASSIC",encoding={'time':{'units':'days since 1900-01-01'}})

/var/folders/6r/n_mpcj7d7pdf9ncqvjy5vb8c0000jm/T/ipykernel_35818/664318075.py:1: UserWarning: Times can't be serialized faithfully to int64 with requested units 'days since 1900-01-01'. Resolution of 'hours' needed. Serializing times to floating point instead. Set encoding['dtype'] to integer dtype to serialize to int64. Set encoding['dtype'] to floating point dtype to silence this warning.
  wop_nc_qc.to_netcdf(user_data_dir+filename,format="NETCDF3_CLASSIC",encoding={'time':{'units':'days since 1900-01-01'}})


In [134]:
wop_nc_qc

<xarray.Dataset> Size: 280kB
Dimensions:       (time: 8752, depth: 1, latitude: 1, longitude: 1)
Coordinates:
  * time          (time) datetime64[ns] 70kB 2025-08-15T02:00:00 ... 2026-08-...
  * depth         (depth) float64 8B 54.0
  * latitude      (latitude) float64 8B 62.2
  * longitude     (longitude) float64 8B -174.7
Data variables:
    temperature   (time, depth, latitude, longitude) float64 70kB -1.446 ... ...
    conductivity  (time, depth, latitude, longitude) float64 70kB 2.553 ... 2...
    salinity      (time, depth, latitude, longitude) float64 70kB 31.83 ... 3...
Attributes: (12/33)
    Latitude_DegMMddW:                      62 12.106 N
    Longitude_DegMMddN:                     174 39.673 W
    MooringID:                              25BS-8A
    platform_deployment_date:               2025-08-15T01:18:00Z
    platform_deployment_cruise_name:        SKQ202515S
    platform_recovery_date:                 2026-08-14T17:00:01Z
    ...                                     ...
    InstrumentSerialNumber:                 1858
    InstrumentType:                         SBE-37
    date_created:                           2026-09-13T16:23:26Z
    date_modified:                          2026-10-29 13:29:47.155557
    QC_indicator:                           ProbablyGood
    history:                                Trimmed to deployment.\nQC'd: 202...